In [ ]:
import pandas as pd
import polars as pl

DATA = "../data/raw/anes_timeseries_2008.dta"
TEXT_RESPONSES_FILE = (
    "../data/raw/anes_timeseries_2008_openends_redacted_Dec2012Revision.xls"
)
RELEVANT_METADATA_COLUMNS = [
    "V080001",  # Case ID
    "V081001",  # If respondent was interviewed post-election as well (needed because these are the ones we have open-endeds for)
    "V083097",  # Party identification
    "V083217",  # Years of education
    "V083215x",  # Age
]

In [ ]:
pd_df = pd.read_stata(DATA)
pd_df = pd_df[RELEVANT_METADATA_COLUMNS]
for column in RELEVANT_METADATA_COLUMNS:
    pd_df[column] = pd_df[column].astype("string")
metadata_df = (
    pl.from_pandas(pd_df)
    .rename(
        {
            "V080001": "id",
            "V081001": "post_election_participation",
            "V083097": "party_id",
            "V083217": "years_of_education",
            "V083215x": "age",
        }
    )
    .with_columns(
        pl.col("id").cast(pl.Float64).cast(pl.Int64),
        pl.when(pl.col("age").is_in(["-9. Refused", "-8. Don't know"]))
        .then(None)
        .otherwise(pl.col("age"))
        .alias("age"),
        pl.when(pl.col("years_of_education").is_in(["-9. Refused", "-8. Don't know"]))
        .then(None)
        .otherwise(pl.col("years_of_education"))
        .alias("years_of_education"),
    )
    .with_columns(
        pl.col("age").cast(pl.Float64).cast(pl.Int64),
        pl.col("years_of_education").cast(pl.Float64).cast(pl.Int64),
    )
)
metadata_df

In [ ]:
text_df = (
    pl.read_excel(TEXT_RESPONSES_FILE, sheet_name="MIPpolit1")[1:]
    .rename(
        {
            "caseID": "id",
            "Q3b1. CSES_ISSPOLITICAL1 (CSES Module: Most Important Political Issue)": "most_important_issue",
        }
    )
    .drop("post-election IW")
    .with_columns(pl.col("id").cast(pl.Int64))
)
text_df

In [ ]:
merged_df = text_df.join(on="id", other=metadata_df)
merged_df

In [ ]:
merged_df = text_df.join(on="id", other=metadata_df)

party_map = {
    "1. Democrat": "Democrat",
    "2. Republican": "Republican",
    "3. Independent": "Independent",
    "4. Other party (SPECIFY)": "Other",
    "5. No preference {VOL}": "No Preference",
    "-8. Don't know": "Unknown",
    "-9. Refused": "Refused",
}

merged_df = merged_df.with_columns(
    pl.col("post_election_participation")
    .str.slice(0, 1)
    .cast(pl.Int8)
    .cast(pl.Boolean),
    pl.col("party_id").replace(party_map),
)
merged_df

In [ ]:
embeddings = "../data/processed/anes_embeddings.parquet"
embeddings_df = pl.read_parquet(embeddings)
embeddings_df